# Phương pháp tiếp cận
### 1. Logistic Regression (Trường phái Thống kê)
* **Lý do sử dụng:** Đây là "ông vua" trong ngành ngân hàng (Traditional Credit Scoring).
* **Ưu điểm:** Cực kỳ dễ giải thích. Hệ số (coefficients) của mô hình cho biết chính xác một đơn vị thu nhập tăng lên thì rủi ro giảm bao nhiêu.
* **Vai trò:** Làm **Baseline Model** (mốc đối chứng). Nếu các mô hình phức tạp hơn không thắng được Logistic Regression đáng kể, ta nên chọn nó để dễ báo cáo nghiệp vụ.

### 2. Random Forest (Trường phái Ensemble - Bagging)
* **Lý do sử dụng:** Hoạt động cực tốt trên dữ liệu bảng (tabular data) và không nhạy cảm với các biến có phân phối lệch hoặc Outliers.
* **Ưu điểm:** Khả năng bắt được các mối quan hệ phi tuyến (ví dụ: thu nhập thấp + điểm tín dụng thấp = rủi ro tăng vọt).
* **Vai trò:** Mô hình ổn định, giúp kiểm tra tầm quan trọng của các đặc trưng (Feature Importance).

### 3. XGBoost / LightGBM (Trường phái Boosting)
* **Lý do sử dụng:** Hiện tại là "vũ khí hạng nặng" nhất cho dữ liệu cấu trúc.
* **Ưu điểm:** Tốc độ huấn luyện nhanh, có khả năng tự xử lý dữ liệu mất cân bằng (imbalanced data) thông qua tham số `scale_pos_weight`.
* **Vai trò:** **Challenger Model** (Kẻ thách thức). Chúng ta kỳ vọng mô hình này sẽ cho chỉ số ROC-AUC cao nhất.

---

### 4. Chiến lược huấn luyện (Training Strategy)

Để đảm bảo các thuật toán này "so găng" công bằng, mình sẽ áp dụng:

1.  **Cross-Validation (5-Fold):** Chia dữ liệu tập Train thành 5 phần, huấn luyện 5 lần để đảm bảo kết quả không phải do may mắn.
2.  **Hyperparameter Tuning:** Sử dụng `RandomizedSearchCV` để tìm bộ thông số (như độ sâu của cây, tỷ lệ học) tối ưu nhất cho từng loại quân bài trên.
3.  **Handling Imbalance:** Vì nợ xấu thường ít, mình sẽ sử dụng kỹ thuật **Weighting** (trọng số) để mô hình tập trung học các ca Default kỹ hơn.

In [11]:
# Install package builtin
!pip install xgboost

# 1. Khởi tạo dữ liệu

In [12]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score
from sklearn.model_selection import cross_val_score

# Load dữ liệu cuối cùng
def load_data_bundle(file_path='./data/processed/final_feature_bundle.joblib'):
    """
    Load toàn bộ tập Train/Test từ file joblib và in ra kích thước kiểm tra.
    """
    print(file_path)
    if not os.path.exists(file_path):
        print(f"❌ Không tìm thấy file tại: {file_path}")
        return None
    
    # Load bundle
    data = joblib.load(file_path)
    
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']
    
    # Kiểm tra kích thước
    print("--- KIỂM TRA KÍCH THƯỚC DỮ LIỆU ---")
    print(f"Shape X_train: {X_train.shape} | Shape y_train: {y_train.shape}")
    print(f"Shape X_test:  {X_test.shape}  | Shape y_test:  {y_test.shape}")
    
    # Kiểm tra tỷ lệ nợ xấu để đảm bảo Stratified Split hoạt động đúng
    train_default_rate = (y_train.sum() / len(y_train) * 100).round(2)
    test_default_rate = (y_test.sum() / len(y_test) * 100).round(2)
    
    print(f"\n--- TỶ LỆ NỢ XẤU (DEFAULT RATE) ---")
    print(f"Trong tập Train: {train_default_rate}%")
    print(f"Trong tập Test:  {test_default_rate}%")
    
    return X_train, X_test, y_train, y_test

In [13]:
# # load dữ liệu
# X_train, X_test, y_train, y_test = load_data_bundle('./data/processed/final_feature_bundle.joblib')

In [ ]:
def final_preprocessing(df):
    """Đảm bảo 100% dữ liệu là số trước khi Training/EDA."""
    df_transformed = df.copy()
    
    # 1. Chuyển đổi các cột Yes/No sang 1/0
    binary_cols = [col for col in df_transformed.columns if col.startswith('Has')]
    for col in binary_cols:
        if df_transformed[col].dtype == 'object':
            df_transformed[col] = df_transformed[col].map({'Yes': 1, 'No': 0, 'Y': 1, 'N': 0})
    
    # 2. Xử lý các cột phân loại còn lại (nếu chưa xử lý ở bước 04)
    # Chỉ chọn các cột kiểu 'object' để One-Hot
    cat_cols = df_transformed.select_dtypes(include=['object']).columns.tolist()
    if len(cat_cols) > 0:
        df_transformed = pd.get_dummies(df_transformed, columns=cat_cols, drop_first=True)
        
    return df_transformed

# # Thực thi cập nhật dữ liệu
# X_train = final_preprocessing(X_train)
# X_test = final_preprocessing(X_test)

# 2. Hàm Huấn luyện và Đánh giá sơ bộ (Baseline Comparison)

In [15]:
# Làm sạch dữ liệu trước khi thực hiện training
# Đối với các giải thuật: Logictic Regression -> Không xử lý được dữ liệu có giá trị NaN
# Đối với các giải thuật: Decision Tree như RF, XGBoost xử lý được dữ liệu NaN
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

def super_clean_data(df):
    """Xử lý triệt để NaN, String và Scaling để tất cả các model đều chạy được."""
    df_clean = df.copy()
    
    # 1. Chuyển đổi Binary String ('Yes'/'No') sang 1/0
    binary_mapping = {'Yes': 1, 'No': 0, 'Y': 1, 'N': 0}
    for col in df_clean.select_dtypes(include=['object']).columns:
        unique_vals = set(df_clean[col].dropna().unique())
        if unique_vals.issubset(set(binary_mapping.keys())):
            df_clean[col] = df_clean[col].map(binary_mapping)

    # 2. Điền giá trị thiếu (NaN)
    # Tách cột số và cột chữ để điền median/mode chính xác
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
    
    cat_cols = df_clean.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if not df_clean[col].mode().empty:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

    # 3. One-Hot Encoding cho các biến phân loại còn lại
    df_clean = pd.get_dummies(df_clean, drop_first=True)

    # 4. Scaling (Quan trọng cho Logistic Regression)
    # Chúng ta scale toàn bộ biến số để các model dựa trên khoảng cách chạy tốt hơn
    scaler = StandardScaler()
    # Lưu ý: Chỉ scale các cột features, không scale biến mục tiêu (nếu có trong df)
    cols_to_scale = df_clean.select_dtypes(include=[np.number]).columns
    # Loại bỏ biến Default nếu nó lỡ nằm trong df_clean
    cols_to_scale = [c for c in cols_to_scale if c != 'Default']
    
    df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])
    
    print("✅ Clean & Scale dữ liệu hoàn tất.")
    return df_clean


In [16]:
import os
import joblib
from datetime import datetime
import pandas as pd

def save_model_with_benchmark(model, model_name, auc_score, folder_path='../models'):
    """
    Lưu model kèm versioning và cập nhật file benchmark để theo dõi hiệu năng.
    """
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    
    # 1. Tạo tên file định dạng: model_XGBoost_AUC0.85_20231027.joblib
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    auc_str = f"{auc_score:.4f}".replace(".", "")
    filename = f"{model_name}_AUC{auc_str}_{timestamp}.joblib"
    full_path = os.path.join(folder_path, filename)
    
    # 2. Lưu file vật lý
    joblib.dump(model, full_path)
    
    # 3. Cập nhật file Benchmark (CSV) để theo dõi lịch sử
    benchmark_file = os.path.join(folder_path, 'model_benchmark_history.csv')
    new_record = {
        'timestamp': timestamp,
        'model_name': model_name,
        'auc_score': auc_score,
        'file_path': filename
    }
    
    if os.path.exists(benchmark_file):
        history_df = pd.read_csv(benchmark_file)
        history_df = pd.concat([history_df, pd.DataFrame([new_record])], ignore_index=True)
    else:
        history_df = pd.DataFrame([new_record])
        
    history_df.to_csv(benchmark_file, index=False)
    
    print(f"✨ Model saved: {filename}")
    print(f"📊 Benchmark history updated in: {benchmark_file}")
    return filename

# --- CÁCH SỬ DỤNG TRONG BƯỚC 07 ---
# Giả sử bạn chọn XGBoost sau khi so sánh:
# best_auc = 0.865
# save_model_with_benchmark(best_xgb_model, "XGBoost_Baseline", best_auc)

In [17]:
def train_and_compare_models(X, y, folder_path='./models'):
    """
    Huấn luyện, so sánh và tự động lưu Benchmark cho từng model.
    """
    if X.isnull().sum().sum() > 0:
        print("❌ Lỗi: Dữ liệu vẫn còn NaN. Hãy chạy super_clean_data trước.")
        return None

    # Xử lý mất cân bằng
    ratio = (len(y) - sum(y)) / (sum(y) + 1e-6) # Tránh chia cho 0
    
    # Gợi ý: Dùng Snake Case cho tên model để lưu file an toàn
    models = {
        'Logistic_Regression': LogisticRegression(max_iter=100, class_weight='balanced', random_state=42),
        'Random_Forest': RandomForestClassifier(n_estimators=20, class_weight='balanced', random_state=42),
        'XGBoost': XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=42)
    }
    
    results = []
    
    for name, model in models.items():
        try:
            # 1. Đánh giá tính ổn định (Cross Validation)
            cv_auc = cross_val_score(model, X, y, cv=5, scoring='roc_auc').mean()
            
            # 2. Huấn luyện thực tế
            model.fit(X, y)
            
            # 3. Lưu Model & Benchmark (Hàm của bạn gọi ở đây là cực chuẩn)
            # Hàm này sẽ tự động ghi log vào file CSV
            save_model_with_benchmark(model, name, cv_auc, folder_path=folder_path)
            
            results.append({
                'Model': name,
                'CV ROC-AUC': cv_auc,
                'Object': model
            })
            print(f"✅ Finished: {name:20} | CV AUC: {cv_auc:.4f}")
            
        except Exception as e:
            print(f"❌ Error training {name}: {e}")
            
    return pd.DataFrame(results)

In [20]:
# --- LUỒNG THỰC THI ---
# Bước 1: Làm sạch và chuẩn hóa dữ liệu
# 1. Load dữ liệu gốc từ bundle
file_path = './data/processed/final_feature_bundle.joblib'
X_train, X_test, y_train, y_test = load_data_bundle(file_path)

# print("🔄 Đang tiến hành làm sạch dữ liệu toàn diện...")
# 
# # 2. Thực hiện làm sạch (Xử lý NaN, String, Scaling)
# X_train_final = super_clean_data(X_train)
# X_test_final = super_clean_data(X_test)
# 
# # 3. Đảm bảo số lượng cột giữa Train và Test phải khớp nhau sau khi One-Hot Encoding
# # (Trường hợp tập Test thiếu một số giá trị phân loại mà tập Train có)
# X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)
# 
# # 4. Đóng gói lại vào Dictionary với đúng cấu trúc cũ
# clean_data_bundle = {
#     'X_train': X_train_final,
#     'X_test': X_test_final,
#     'y_train': y_train,
#     'y_test': y_test
# }
# 
# # 5. Ghi đè lại file cũ để các bước sau (08, 09) load lên là dùng được ngay
# joblib.dump(clean_data_bundle, file_path)
# print(f"✅ Đã lưu dữ liệu SẠCH (NaN=0, Numeric only) vào: {file_path}")

./data/processed/final_feature_bundle.joblib
--- KIỂM TRA KÍCH THƯỚC DỮ LIỆU ---
Shape X_train: (204277, 22) | Shape y_train: (204277,)
Shape X_test:  (51070, 22)  | Shape y_test:  (51070,)

--- TỶ LỆ NỢ XẤU (DEFAULT RATE) ---
Trong tập Train: 11.61%
Trong tập Test:  11.61%


In [21]:
# Bước 2: Train và so sánh
model_performance = train_and_compare_models(X_train_final, y_train)

if model_performance is not None:
    display(model_performance.sort_values(by='CV ROC-AUC', ascending=False))

✨ Model saved: Logistic_Regression_AUC07458_20260319_0934.joblib
📊 Benchmark history updated in: ./models\model_benchmark_history.csv
✅ Finished: Logistic_Regression  | CV AUC: 0.7458
✨ Model saved: Random_Forest_AUC06870_20260319_0935.joblib
📊 Benchmark history updated in: ./models\model_benchmark_history.csv
✅ Finished: Random_Forest        | CV AUC: 0.6870
✨ Model saved: XGBoost_AUC07328_20260319_0935.joblib
📊 Benchmark history updated in: ./models\model_benchmark_history.csv
✅ Finished: XGBoost              | CV AUC: 0.7328


,Model,CV ROC-AUC,Object
0,Logistic_Regression,0.745792,"LogisticRegression(class_weight='balanced', ra..."
2,XGBoost,0.732779,"XGBClassifier(base_score=None, booster=None, c..."
1,Random_Forest,0.686992,"(DecisionTreeClassifier(max_features='sqrt', r..."


### Nhận xét

1. Phân tích thứ hạng (Benchmark)

- Logistic Regression (0.7458) – Xuất sắc! Dữ liệu của có tính tuyến tính mạnh. Việc StandardScaler chuẩn xác đã giúp model này tối ưu nhất.
- XGBoost (0.7328) – Thấp hơn một chút, có thể do chưa tuning (tinh chỉnh) tham số hoặc bị nhiễu (noise).
- Random Forest (0.6870) – Thấp do số lượng cây quá ít (n_estimators=20).

2. Ý nghĩa kinh doanh (Business Insight)
- Mô hình hiện tại (AUC ~0.75) đã ở mức Khá. Nó có khả năng phân biệt nợ xấu tốt hơn nhiều so với việc đoán ngẫu nhiên.

=> Ưu tiên dùng Logistic Regression cho báo cáo vì tính minh bạch và hiệu năng cao nhất hiện tại.